<!-- TARGET: public course-materials repo, sessions/m06/demo.ipynb (follow-along: the Your turn cells are blank). Copy to work/m06/ before editing. -->
# Module 6 · Live demo

**Python for Data Science · ESADE MIBA · Wed 18 Nov 2026**

Operations asks: *where, and when, are deliveries late?* A table of 27 states and 20 months is
540 numbers. Nobody reads that. Today we turn tables into charts with plotly express, pick the
chart that fits the question, and check that each chart shows the numbers we think it shows.

Every chart today starts from the **order mart** you built in the M05 lab:
`data/mart/orders_enriched.parquet`, one row per order. If the cell below stops with an
`AssertionError`, copy `sessions/m05/lab-solution.ipynb` to `work/m05/`, Run All, and come back.

In [ ]:
from pathlib import Path

import pandas as pd
import plotly.express as px

RAW = Path("../../data/raw")
MART_FILE = Path("../../data/mart/orders_enriched.parquet")

assert MART_FILE.exists(), (
    "No order mart yet. Copy sessions/m05/lab-solution.ipynb to work/m05/, Run All, then come back."
)
mart = pd.read_parquet(MART_FILE)
assert mart.shape == (99441, 13), (
    f"Your mart has shape {mart.shape}, expected (99441, 13). Run sessions/m05/lab-solution.ipynb to rebuild it."
)

## Hook · Four datasets, one summary

Anscombe's quartet: four small datasets of 11 points each. Here is the summary of each.

In [ ]:
anscombe = pd.DataFrame(
    {
        "dataset": ["I"] * 11 + ["II"] * 11 + ["III"] * 11 + ["IV"] * 11,
        "x": [10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5] * 3 + [8, 8, 8, 8, 8, 8, 8, 19, 8, 8, 8],
        "y": [8.04, 6.95, 7.58, 8.81, 8.33, 9.96, 7.24, 4.26, 10.84, 4.82, 5.68]
        + [9.14, 8.14, 8.74, 8.77, 9.26, 8.10, 6.13, 3.10, 9.13, 7.26, 4.74]
        + [7.46, 6.77, 12.74, 7.11, 7.81, 8.84, 6.08, 5.39, 8.15, 6.42, 5.73]
        + [6.58, 5.76, 7.71, 8.84, 8.47, 7.04, 5.25, 12.50, 5.56, 7.91, 6.89],
    }
)
correlations = {name: group["x"].corr(group["y"]) for name, group in anscombe.groupby("dataset")}
summary = anscombe.groupby("dataset").agg(
    mean_x=("x", "mean"),
    mean_y=("y", "mean"),
    std_x=("x", "std"),
    std_y=("y", "std"),
)
summary.assign(corr_xy=pd.Series(correlations)).round(2)

Same means, same spreads, same correlation, to two decimals. *Predict:* what do the four
scatter plots look like?

In [ ]:
px.scatter(anscombe, x="x", y="y", facet_col="dataset")

A line, a curve, a line with one outlier, and no relationship at all except one point. The
summary was correct and told us nothing. **Plot before you conclude.**

## 1. The plotly express grammar

Every `px` call has the same shape: **a table, and columns mapped to channels.**

```python
px.line(table, x="column", y="column", color="column", ...)
```

One row of the table becomes one point (or bar). So the first job is always a table at the
right grain.

### Is the business growing?

In [ ]:
monthly = (
    mart.groupby("purchase_month")
    .agg(orders=("order_id", "size"))
    .reset_index()
)
monthly = monthly[monthly["purchase_month"].between("2017-01", "2018-08")]
px.line(monthly, x="purchase_month", y="orders")

We keep January 2017 to August 2018: the months before and after hold a handful of orders each.
Why the series ends like that is next week's opening question.

The November 2017 spike is Black Friday (24 Nov).

### Which states bring the revenue?

In [ ]:
by_state = (
    mart.groupby("customer_state")
    .agg(
        orders=("order_id", "size"),
        paid_revenue=("paid_revenue", "sum"),
        item_revenue=("item_revenue", "sum"),
    )
    .reset_index()
)
px.bar(by_state, x="customer_state", y="paid_revenue", hover_data=["orders"])

`hover_data` adds columns to the tooltip: hover over SP. The bars are in alphabetical order,
because that's the order of the rows. We come back to that in block 2.

### More channels: size, colour, name

Delivered orders only. One row per state: orders, median delivery days, share delivered late.

In [ ]:
delivered = mart[mart["order_status"] == "delivered"]
state_delivery = (
    delivered.groupby("customer_state")
    .agg(
        orders=("order_id", "size"),
        median_days=("delivery_days", "median"),
        late_share=("is_late", "mean"),
    )
    .reset_index()
)
state_delivery.head()

The mean of a True/False column is the share of `True`: `late_share` is the share of late
orders.

*Your turn (3 min):* scatter `median_days` (x) against `late_share` (y), with `size` mapped to
`orders` and the state name in the tooltip (`hover_name`). Store it in `fig_states` and show it.

In [ ]:
fig_states = ...
fig_states

Big bubbles (SP, MG, PR) sit bottom-left: fast and rarely late. The North-East (AL, MA, SE) is
slow **and** often late. One chart, four columns: x, y, size, name.

Look bottom-right: AM, AP, RO, AC are among the **slowest** states (median 17–25 days) and the
**least** often late (under 4%). `is_late` compares the delivery with the date promised to the
customer, and those states get generous promises. Slow and late are different questions.

### Long vs wide

`by_state` is **wide**: one column per measure (`paid_revenue`, `item_revenue`). To put the
measure on `color`, plotly needs it as a column of its own: **long** format, one row per
state × measure. `melt` turns wide into long.

*Your turn (2 min):* predict how many rows `revenue_long` will have. Then build it with
`melt`: `id_vars="customer_state"`, the two revenue columns as `value_vars`, `var_name="measure"`,
`value_name="brl"`.

In [ ]:
revenue_long = ...

In [ ]:
revenue_long = by_state.melt(
    id_vars="customer_state",
    value_vars=["paid_revenue", "item_revenue"],
    var_name="measure",
    value_name="brl",
)
print(len(by_state), "states →", len(revenue_long), "rows")
revenue_long.head()

In [ ]:
px.bar(revenue_long, x="customer_state", y="brl", color="measure", barmode="group")

Remember the two revenue numbers: per state they are almost identical. The chart shows it in
one glance.

### Two keys, and a share of the total

Late deliveries **by month**: a `groupby` on two keys gives one row per month × delivery
status. First, a readable label for each order.

In [ ]:
in_range = mart[mart["purchase_month"].between("2017-01", "2018-08")]
in_range = in_range.assign(
    delivery=in_range["is_late"].map({True: "late", False: "on time"}).fillna("not delivered")
)
by_month_delivery = (
    in_range.groupby(["purchase_month", "delivery"]).agg(orders=("order_id", "size")).reset_index()
)
by_month_delivery.head(6)

November 2017 has more late orders than any month before it, but also many more orders. The
fair comparison is the **share** of each month's orders.

`agg` collapses each group to one row. `transform` computes a group number and puts it back on
**every row** of the group, so the table keeps its length.

In [ ]:
print(len(by_month_delivery.groupby("purchase_month")["orders"].sum()), "rows with sum")
print(len(by_month_delivery.groupby("purchase_month")["orders"].transform("sum")), "rows with transform")

*Your turn (3 min):* add a column `share` to `by_month_delivery`: each row's orders divided by
the month's total orders (use `transform`).

In [ ]:
by_month_delivery["share"] = ...

In [ ]:
month_total = by_month_delivery.groupby("purchase_month")["orders"].transform("sum")
by_month_delivery["share"] = by_month_delivery["orders"] / month_total
assert (by_month_delivery.groupby("purchase_month")["share"].sum().round(6) == 1).all()
px.bar(by_month_delivery, x="purchase_month", y="share", color="delivery")

Each month adds up to 100% (the `assert` proves it). Two months stand out: November 2017
(Black Friday) and February–March 2018. Volume alone doesn't explain March: it had about the
same orders as January.

## 2. Distributions and comparisons

| Question | Chart |
|---|---|
| How is one number spread out? | histogram, box |
| How do groups compare? | bar (sorted!), box per group |
| How does it change over time? | line |
| Do two numbers move together? | scatter |

### How long does a delivery take?

In [ ]:
px.histogram(delivered, x="delivery_days")

Most orders arrive within two weeks; a long tail runs past 200 days. The tail squeezes the
interesting part into the left edge. Zoom in (and say so in the title, or the reader won't
know you did):

In [ ]:
px.histogram(
    delivered,
    x="delivery_days",
    range_x=[0, 60],
    nbins=200,
    title="Delivery days, zoomed to 0–60 (a few orders take up to 209)",
)

### Delivery time by state

A box shows the median, the quartiles and the spread of each group. 27 histograms would be too
many to read.

In [ ]:
px.box(delivered, x="customer_state", y="delivery_days", points=False)

Alphabetical order hides the story. `category_orders` sets the order of a categorical axis.

*Your turn (2 min):* make `states_by_median`, the list of states sorted by `median_days`
(from `state_delivery`), fastest first.

In [ ]:
states_by_median = ...

In [ ]:
states_by_median = state_delivery.sort_values("median_days")["customer_state"].tolist()
px.box(
    delivered,
    x="customer_state",
    y="delivery_days",
    points=False,
    category_orders={"customer_state": states_by_median},
)

Now it reads: SP is fastest, the North (RR, AP, AM) slowest. Same data, one argument.

### Counts vs shares

Operations wants to know **where** lateness is a problem. Count the late orders per state:

In [ ]:
late_by_state = (
    delivered.groupby("customer_state")
    .agg(orders=("order_id", "size"), late_orders=("is_late", "sum"), late_share=("is_late", "mean"))
    .reset_index()
)
px.bar(late_by_state.sort_values("late_orders", ascending=False), x="customer_state", y="late_orders")

SP has the most late orders, RJ close behind. Is SP the problem?

*Your turn (2 min):* the same bar chart, but of `late_share`, sorted from highest to lowest.
Store it in `fig_late_share`.

In [ ]:
fig_late_share = ...
fig_late_share

SP is now near the **bottom**: it has many late orders because it has many orders. A count
answers "where is the volume?"; a share answers "where is the risk?". Choose the one that fits
the question, and say which one you plotted.

## 3. Faceting and polish

### Colour or facets?

Orders per month for the three biggest states and the rest of Brazil.

In [ ]:
in_range = in_range.assign(
    state_group=in_range["customer_state"].where(in_range["customer_state"].isin(["SP", "RJ", "MG"]), "other")
)
by_month_state = (
    in_range.groupby(["purchase_month", "state_group"]).agg(orders=("order_id", "size")).reset_index()
)
px.line(by_month_state, x="purchase_month", y="orders", color="state_group")

`color`: one panel, good for comparing **levels**. `facet_col`: one panel per group, good for
comparing **shapes**. `facet_col_wrap` sets panels per row, and `category_orders` fixes the
order of the panels.

In [ ]:
px.line(
    by_month_state,
    x="purchase_month",
    y="orders",
    facet_col="state_group",
    facet_col_wrap=2,
    category_orders={"state_group": ["SP", "RJ", "MG", "other"]},
)

### Polish: titles, labels, a house style

A chart for an executive needs a title that says the finding, axis labels in words, and no
column names like `late_share`. `labels` renames columns on the chart; `title` sets the title;
`update_layout` changes anything else.

A **template** sets the style once for every chart. Plotly ships several (`plotly_white` is a
clean one). We add the ESADE colours on top:

In [ ]:
import plotly.graph_objects as go
import plotly.io as pio

ESADE_BLUE = "#00205B"
ESADE_LIGHT_BLUE = "#4A9EFF"
pio.templates["esade"] = go.layout.Template(
    layout=dict(
        font=dict(family="Arial", color="#2B2B2B"),
        title=dict(font=dict(color=ESADE_BLUE, size=20)),
        colorway=[ESADE_BLUE, ESADE_LIGHT_BLUE, "#E07A1F", "#7A869A", "#2CA58D", "#C1272D"],
    )
)
pio.templates.default = "plotly_white+esade"

In [ ]:
fig = px.bar(
    late_by_state.sort_values("late_share", ascending=False),
    x="customer_state",
    y="late_share",
    hover_data=["orders", "late_orders"],
    title="In Alagoas and Maranhão up to 1 in 5 deliveries is late; in São Paulo, 1 in 22",
    labels={"customer_state": "Customer state", "late_share": "Share of orders delivered late"},
)
fig.update_layout(yaxis_tickformat=".0%")
fig

*Your turn (3 min):* polish the monthly late-share chart: keep only the `late` rows of
`by_month_delivery`, plot `share` by month as a line, with a title that says the finding, axis
labels in words, and the y axis as a percentage. Store it in `fig_late_month`.

In [ ]:
fig_late_month = ...
fig_late_month

## 4. Check the chart

The running question of the course: **how do you know this number is right?** A chart is a
table drawn as shapes. Its numbers live in `fig.data`, and we can test them like any table.

In [ ]:
fig_revenue = px.bar(by_state.sort_values("paid_revenue", ascending=False), x="customer_state", y="paid_revenue")
plotted = pd.Series(list(fig_revenue.data[0].y), index=list(fig_revenue.data[0].x))
plotted.head()

In [ ]:
expected = mart.groupby("customer_state")["paid_revenue"].sum()
assert (plotted - expected).abs().max() < 0.01
print(f"bars add up to {plotted.sum():,.2f} BRL")

The bars match a fresh `groupby`, and they add up to the 16.0 M paid revenue from M05.

### A colleague's chart

A colleague sends this: *revenue by seller state*. It looks perfectly reasonable.

In [ ]:
items = pd.read_csv(RAW / "olist_order_items_dataset.csv")
payments = pd.read_csv(RAW / "olist_order_payments_dataset.csv")
sellers = pd.read_csv(RAW / "olist_sellers_dataset.csv")

colleague_table = items.merge(payments, on="order_id").merge(
    sellers[["seller_id", "seller_state"]], on="seller_id"
)
colleague_by_state = colleague_table.groupby("seller_state").agg(revenue=("payment_value", "sum")).reset_index()
fig_colleague = px.bar(
    colleague_by_state.sort_values("revenue", ascending=False),
    x="seller_state",
    y="revenue",
    title="Revenue by seller state",
)
fig_colleague

*Your turn (2 min):* add up the bars of `fig_colleague` into `colleague_total`. Compare it with
the paid revenue in `payments`.

In [ ]:
colleague_total = ...

In [ ]:
colleague_total = sum(fig_colleague.data[0].y)
print(f"chart: {colleague_total:,.2f} BRL")
print(f"paid:  {payments['payment_value'].sum():,.2f} BRL")

**20.3 M.** The fan-out from M05: items × payments counts each payment once per item line. The
chart gives no hint of it; only the check does.

The fix is M05's rule: to split by seller, use **item revenue**, which lives on the item line.

In [ ]:
items = items.assign(item_revenue=items["price"] + items["freight_value"])
items_with_seller = items.merge(
    sellers[["seller_id", "seller_state"]], on="seller_id", how="left", validate="many_to_one"
)
by_seller_state = items_with_seller.groupby("seller_state").agg(item_revenue=("item_revenue", "sum")).reset_index()
fig_fixed = px.bar(
    by_seller_state.sort_values("item_revenue", ascending=False),
    x="seller_state",
    y="item_revenue",
    title="São Paulo sellers bring almost two thirds of item revenue",
    labels={"seller_state": "Seller state", "item_revenue": "Item revenue (BRL)"},
)
assert abs(sum(fig_fixed.data[0].y) - items["item_revenue"].sum()) < 0.01
fig_fixed

## Today in one table

| Step | Check |
|---|---|
| Build the table first | one row = one point or bar; know its grain |
| Map columns to channels | `x`, `y`, `color`, `size`, `hover_name`, `hover_data` |
| Long, not wide | `melt` so the measure is a column |
| Shares | `groupby(...).transform("sum")` keeps the rows; assert shares add to 1 |
| Pick the chart | histogram/box for spread, sorted bar to compare, line over time, scatter for pairs |
| Count or share? | volume vs risk: say which one you plotted |
| Polish | title says the finding, `labels` in words, `category_orders`, template |
| Check the chart | `fig.data[0].y` against a fresh `groupby`, totals against the source |